# FinCast Paper-Style MoE Benchmark

This notebook is a paper-oriented zero-shot test harness for the released FinCast v1 checkpoint.

What it does:
- evaluates the released checkpoint on rolling holdout windows from your CSV
- keeps the default paper-aligned checkpoint setting (`num_experts=4`, `top_n=2`) as the baseline
- leaves a runtime patch interface for MoE routing experiments
- captures expert routing load so you can inspect whether a MoE change is actually doing something

Important limitation:
- `num_experts` is structurally tied to the released checkpoint and is not hot-swappable here
- safe runtime experiments in this notebook are mainly `gating_top_n`, `threshold_train`, `threshold_eval`, `capacity_factor_*`, and routing-loss coefficients


In [ ]:
from pathlib import Path
import sys

WORKSPACE_ROOT = Path.cwd().resolve()
FINCAST_SRC = WORKSPACE_ROOT / "FinCast-fts" / "src"
LAB_DIR = WORKSPACE_ROOT / "experiments" / "fincast_paper_lab"
for p in (FINCAST_SRC, LAB_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"WORKSPACE_ROOT={WORKSPACE_ROOT}")
print(f"FINCAST_SRC={FINCAST_SRC}")
print(f"LAB_DIR={LAB_DIR}")


In [ ]:
import pandas as pd
import torch

from paper_moe_lab import (
    apply_moe_runtime_patch,
    build_backtest_windows,
    compare_experiments,
    load_fincast_api,
    make_delta_table,
    make_model_config,
    plot_routing_summary,
    plot_window_forecast,
    run_experiment,
    summarize_moe_layers,
    windows_to_frame,
)

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device={torch.cuda.get_device_name(0)}")


In [ ]:
# Test data (moved from FinCast-fts/Inference/sample_close_monthly.csv → data/raw/).
# Replace DATA_PATH with another CSV if you want to benchmark a different series.

MODEL_PATH = WORKSPACE_ROOT / "models" / "FinCast" / "v1.pth"
DATA_PATH = WORKSPACE_ROOT / "data" / "raw" / "sample_close_monthly.csv"

# Leave TARGET_COLUMNS as None to auto-select numeric columns.
TARGET_COLUMNS = None

# Paper-style baseline for the released v1 checkpoint.
PAPER_BASELINE = {
    "backend": "gpu",
    "context_len": 128,
    "horizon_len": 32,
    "per_core_batch_size": 32,
    "forecast_mode": "mean",
    "num_experts": 4,
    "gating_top_n": 2,
    "threshold_train": 0.2,
    "threshold_eval": 0.2,
    "load_from_compile": True,
}

# Rolling holdout test settings.
EVAL_CONFIG = {
    "windows_per_series": 3,
    "stride": 32,
    "frequency_code": 0,
}

# Runtime MoE patch interface.
# Safe hot-swap knobs for the released checkpoint are listed here.
MOE_PATCH = {
    "enabled": True,
    "layer_indices": "all",  # or e.g. [0, 1, 2, 3]
    "gating_top_n": 2,
    "threshold_train": 0.2,
    "threshold_eval": 0.2,
    "capacity_factor_train": 1.25,
    "capacity_factor_eval": 2.0,
    "balance_loss_coef": 1e-2,
    "router_z_loss_coef": 1e-3,
    "num_experts": None,  # placeholder only; changing this is not supported for the released v1 checkpoint
}

assert MODEL_PATH.exists(), f"Missing model checkpoint: {MODEL_PATH}"
assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH}"
print(f"MODEL_PATH={MODEL_PATH}")
print(f"DATA_PATH={DATA_PATH}")


In [ ]:
# Optional sanity check: inspect MoE structure before full evaluation.
baseline_config = make_model_config(model_path=MODEL_PATH, **PAPER_BASELINE)
model, api = load_fincast_api(baseline_config)
display(summarize_moe_layers(model).head())
if MOE_PATCH["enabled"]:
    display(apply_moe_runtime_patch(model, MOE_PATCH).head())
del model, api


In [ ]:
windows = build_backtest_windows(
    DATA_PATH,
    context_len=PAPER_BASELINE["context_len"],
    horizon_len=PAPER_BASELINE["horizon_len"],
    target_columns=TARGET_COLUMNS,
    windows_per_series=EVAL_CONFIG["windows_per_series"],
    stride=EVAL_CONFIG["stride"],
    frequency_code=EVAL_CONFIG["frequency_code"],
)

display(windows_to_frame(windows).head())
print(f"num_windows={len(windows)}")


In [ ]:
baseline_result = run_experiment(
    experiment_name="paper_baseline",
    model_config=baseline_config,
    windows=windows,
    moe_patch=None,
    forecast_mode=PAPER_BASELINE["forecast_mode"],
    capture_routing=True,
)

display(baseline_result["summary_df"])
print(f"baseline_runtime_sec={baseline_result['runtime_sec']:.4f}")


In [ ]:
patched_result = None
if MOE_PATCH["enabled"]:
    patched_result = run_experiment(
        experiment_name="moe_patched",
        model_config=baseline_config,
        windows=windows,
        moe_patch=MOE_PATCH,
        forecast_mode=PAPER_BASELINE["forecast_mode"],
        capture_routing=True,
    )
    display(patched_result["summary_df"])
    print(f"patched_runtime_sec={patched_result['runtime_sec']:.4f}")


In [ ]:
if patched_result is not None:
    display(compare_experiments(baseline_result, patched_result))
    display(make_delta_table(baseline_result, patched_result))
else:
    display(baseline_result["summary_df"])


In [ ]:
# Plot one forecast window from the baseline.
plot_window_forecast(baseline_result)

# Plot the patched forecast and routing summary if the patch is enabled.
if patched_result is not None:
    plot_window_forecast(patched_result)
    plot_routing_summary(patched_result["routing_df"])
